In [17]:
import subprocess

def sh(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
    return result

In [18]:
%%writefile script.py

import json
import os
import random
import pandas as pd

random.seed(42)

VALID_DOMAINS = ["gmail.com", "yahoo.com", "edu.in"]
INCORRECT_EMAILS = ["cat@apple.com", "dog@banana.com"]
NAME = ["SAM", "TAM", "CAT", "PAT"]

NUM_SHARDS = 8
ROWS_PER_SHARD = 10

def make_valid_email(user_id):
    return f"user{user_id}@{random.choice(VALID_DOMAINS)}"

os.makedirs("shards", exist_ok=True)

ground_truth = {}

for shard_idx in range(NUM_SHARDS):
    n_invalid = random.randint(1, 4)
    invalid_positions = set(random.sample(range(ROWS_PER_SHARD), n_invalid))
    names, emails = [], []
    for row_idx in range(ROWS_PER_SHARD):
        user_id = shard_idx * ROWS_PER_SHARD + row_idx
        if row_idx in invalid_positions:
            if random.random() < 0.5:
                names.append(random.choice(NAME))
                emails.append(random.choice(INCORRECT_EMAILS))
            else:
                names.append("")
                emails.append(make_valid_email(user_id))
        else:
            names.append(random.choice(NAME))
            emails.append(make_valid_email(user_id))

    df = pd.DataFrame({"Name": names, "Email": emails})
    df.to_csv(f"shards/shard_{shard_idx}.csv", index = False)
    ground_truth[f"shard_{shard_idx}"] = n_invalid

    with open("shards/ground_truth.json", "w") as f:
        json.dump(ground_truth, f)

Overwriting script.py


In [19]:
%%writefile validate_shard.py
import json
import os
import re
import sys
import pandas as pd

EMAIL_RE = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")

def is_valid(name, email):
    if name == None or name == "":
        return False
    if email == None or not EMAIL_RE.match(str(email).strip()):
        return False
    return True

def main():
    index = int(os.environ.get("JOB_COMPLETION_INDEX"))
    pod_name = os.environ.get("POD_NAME", "unknown_pod")
    node_name = os.environ.get("NODE_NAME", "unknown_node")
    shard_path = f"/data/shard_{index}.csv"
    df = pd.read_csv(shard_path)
    invalid_rows = sum([0 if is_valid(row["Name"], row["Email"]) else 1 for _, row in df.iterrows()])
    results = {"shard_idx" : index, "shard_file" : f"shard_{index}.csv", "pod_name": pod_name, "node_name":node_name, "total_rows": len(df), "invalid_rows": invalid_rows}

if ___name__ == "__main__":
    main()

Overwriting validate_shard.py


In [20]:
%%writefile requirements.txt
pandas

Overwriting requirements.txt


In [26]:
%%writefile Dockerfile
FROM python:3.11

WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY validate_shard.py .
COPY shards/*.csv /data/

CMD ["python3", "validate_shard.py"]

Overwriting Dockerfile


In [22]:
%%writefile validate-job.yaml
apiVersion: batch/v1
kind: Job
metadata:
  name: signup-shard-validator
  labels:
    app: signup-shard-validator
spec:
  completions: 8
  parallelism: 4
  completionMode: Indexed
  backoffLimit: 4
  activeDeadlineSeconds: 600
  template:
    metadata:
      labels:
        app: signup-shard-validator
    spec:
      restartPolicy: Never
      containers:
      - name: validator
        image: shared-validator:latest
        imagePullPolicy: IfNotPresent
        env:
        - name: POD_NAME
          valueFrom:
            fieldRef:
              fieldPath: metadata.name
        - name: NODE_NAME
          valueFrom:
            fieldRef:
              fieldPath: spec.nodeName
        resources:
          requests:
            cpu: "500m"
            memory: "64Mi"
          limits:
            cpu: "500m"
            memory: "128Mi"


Overwriting validate-job.yaml


In [28]:
%%writefile collect_results.py
import json
import re
from kubernetes import client, config

RESULTS_RE = re.compile(r"RESULT_JSON (\{.*\})")

def main():
    config.load_kube_config()
    v1 = client.CoreV1Api()

    pods = v1.list_namespaced_pod(namespace =' default', label_selector = 'app=signup-shard-validator')

    results = []
    for pod in pods.items:
        pod_name = pod.metadata.name
        node_name = pod.spec.node_name
        log = v1.read_namespaced_pod_log(name = pod_name, namespace = "default")
        match = RESULT_RE.search(log)
        result["node_name_from_api"] = node_name
        results.append(result)
        with open("shards/collected_results.json", "w") as f:
            json.dump(results, f)

if __name__ == "__main__":
    main()

Overwriting collect_results.py
